# Shofly Media Worker
Six sequential Python cells. Run the Telegram preflight before processing authorized media. Store credentials in Colab Secrets.


## 1. Environment and latest repository


In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/MohammedTeir/shofly-project.git"
BRANCH = "main"
REPO_DIR = "/content/shofly"
WORKER_DIR = f"{REPO_DIR}/workers/media-worker"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Updating Shofly repository...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
elif os.path.exists(REPO_DIR):
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    print("Cloning Shofly repository...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "httpx==0.28.1", "pydantic-settings==2.7.0", "telethon>=1.36,<2"], check=True)
if WORKER_DIR not in sys.path:
    sys.path.insert(0, WORKER_DIR)
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "aria2", "ffmpeg"], check=True)
print("Repository ready:", REPO_DIR)
print("Worker directory:", WORKER_DIR)
subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], check=True)


## 2. Download configuration


In [ ]:
from pathlib import Path

USE_ARIA2 = False
TRUSTED_SOURCE_HOSTS = ""

if USE_ARIA2:
    from google.colab import userdata, drive
    TRUSTED_SOURCE_HOSTS = (userdata.get("TRUSTED_SOURCE_HOSTS") or "").strip()
    if not TRUSTED_SOURCE_HOSTS:
        raise ValueError("Set TRUSTED_SOURCE_HOSTS in Colab Secrets.")
    drive.mount("/content/drive")
    storage = Path("/content/drive/MyDrive/Shofly/downloads")
    storage.mkdir(parents=True, exist_ok=True)
    os.environ["DOWNLOADER"] = "aria2c"
    os.environ["ALLOWED_HOSTS"] = TRUSTED_SOURCE_HOSTS
    os.environ["PERSISTENT_DOWNLOAD_DIR"] = str(storage)
    os.environ["ARIA2_CONNECTIONS"] = "4"
    os.environ["ARIA2_RETRIES"] = "3"
    print("aria2c downloader selected.")
else:
    os.environ["DOWNLOADER"] = "httpx"
    os.environ["HTTPX_PARALLEL_CONNECTIONS"] = "8"
    os.environ["HTTPX_CHUNK_SIZE_MB"] = "8"
    from google.colab import drive
    drive.mount("/content/drive")
    storage = Path("/content/drive/MyDrive/Shofly/downloads")
    storage.mkdir(parents=True, exist_ok=True)
    os.environ["PERSISTENT_DOWNLOAD_DIR"] = str(storage)
    os.environ.pop("ALLOWED_HOSTS", None)
    print("HTTPX downloader selected; quality-resume originals retained in Google Drive.")


## 3. Secrets and provider


In [ ]:
from google.colab import userdata
import httpx
from shofly_worker.config import WorkerSettings
from shofly_worker.client import BackendClient
from shofly_worker.providers import LocalDirProvider, TelegramProvider, MTProtoTelegramProvider

def read_secret(name):
    value = userdata.get(name)
    if value is None or not str(value).strip():
        raise ValueError(f"Missing Colab secret: {name}")
    return str(value).strip()

os.environ["API_BASE_URL"] = read_secret("SHOFLY_API_URL")
os.environ["WORKER_API_TOKEN"] = read_secret("WORKER_API_TOKEN")
os.environ["DOWNLOAD_MAX_BYTES"] = str(8 * 1024**3)
os.environ["SPLIT_LARGE_MEDIA"] = "true"
os.environ["SEGMENT_MAX_BYTES"] = str(1900 * 1024**2)
os.environ["FFMPEG_VERIFY"] = "true"
os.environ["FFMPEG_VERIFY_TIMEOUT"] = "600"
os.environ["PROVIDER"] = "telegram_mtproto"
os.environ["TELEGRAM_BOT_TOKEN"] = read_secret("TELEGRAM_BOT_TOKEN")
os.environ["TELEGRAM_CHAT_ID"] = read_secret("TELEGRAM_STORAGE_CHANNEL_ID")
os.environ["TELEGRAM_API_ID"] = read_secret("TELEGRAM_API_ID")
os.environ["TELEGRAM_API_HASH"] = read_secret("TELEGRAM_API_HASH")
os.environ["TELEGRAM_SESSION"] = "/content/shofly_telegram_bot"
os.environ["MTPROTO_PARALLEL_WORKERS"] = "16"
os.environ["MTPROTO_CONNECTIONS"] = "4"
os.environ["GENERATE_QUALITIES"] = "true"
os.environ["TRANSCODE_TIMEOUT"] = "7200"

S = WorkerSettings()
client = BackendClient(S.api_base_url, S.worker_api_token, S.worker_id)
print("Backend:", httpx.get(S.api_base_url + "/health/live", timeout=15).json())
print("Worker token accepted:", client.ping())
print("Worker ID:", S.worker_id)

if S.provider == "telegram_mtproto":
    provider = MTProtoTelegramProvider(S.telegram_api_id, S.telegram_api_hash, S.telegram_bot_token, S.telegram_chat_id, S.telegram_session, S.mtproto_max_bytes, S.mtproto_parallel_workers, S.mtproto_connections)
elif S.provider == "telegram":
    provider = TelegramProvider(S.telegram_bot_token, S.telegram_chat_id, min(S.max_bytes, 50 * 1024**2))
else:
    provider = LocalDirProvider(S.local_dir)
print("Provider:", type(provider).__name__)
print("Session configured:", bool(S.telegram_session))


## 4. Telegram preflight (no upload)


In [ ]:
import logging
import inspect

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)
print("Checking Telegram connection...")
print("Bot authentication fix loaded:", "await client.connect()" in inspect.getsource(type(provider)._run))

async def check_telegram(client):
    me = await client.get_me()
    channel = await client.get_entity(provider.chat_id)
    return {"connected": True, "is_bot": bool(me and me.bot), "channel_access": True, "channel_type": type(channel).__name__}

try:
    result = provider._run(check_telegram)
    print("Telegram test:", result)
except Exception as exc:
    print("Telegram test failed.")
    print("Exception type:", type(exc).__name__)
    print("Check bot token, session, channel ID and permissions.")
    raise RuntimeError(f"Telegram preflight failed: {type(exc).__name__}") from None


## 5. Process queued jobs


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)  # avoid logging signed download URLs
from shofly_worker.runner import run_until_empty
dl_http = httpx.Client(timeout=60.0)
results = run_until_empty(client, provider, S, dl_http)
from collections import Counter
print("processed:", len(results), dict(Counter(results)))


## 6. Cleanup


In [ ]:
import shutil
shutil.rmtree(S.tmp_dir, ignore_errors=True)
print("Temporary files cleaned.")
